# Notes 6

The Laplace Transform — Definition and Basic Properties

In [1]:
import numpy as np
import sympy as sym
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, quad
from IPython.display import Math, display
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

## Goals

1.  Understand the **big picture**: what the Laplace transform does and
    why it is useful.

2.  Learn the **definition**
    $\mathcal{L}[x(t)](s) = \int_0^\infty x(t)e^{-st}\,dt$ and compute
    transforms directly from it.

3.  Know the **linearity** of $\mathcal{L}$ and use it to build up
    transforms from a table.

4.  Understand the **inverse transform** $\mathcal{L}^{-1}$ and use
    partial fractions.

5.  Work with the **Heaviside function** to represent piecewise-defined
    forcing.

6.  Apply the **shift property** and **switching property** as
    time-saving tools.

7.  Use all of the above to solve constant-coefficient IVPs with
    discontinuous forcing.

> **Note**
>
> This material corresponds to **Section 3.1** of (Logan 2015).

------------------------------------------------------------------------

## The Big Picture

Before computing anything, it is worth pausing to appreciate *what* the
Laplace transform accomplishes and *why* it is worth learning an
entirely new technique.

The methods we have developed — undetermined coefficients and variation
of parameters — break down when the forcing function $f(t)$ is
**discontinuous** (a switch being flipped, a pulse applied for a finite
time, a sudden impact). These situations are not exotic mathematical
curiosities; they are among the most common scenarios in electrical
engineering, mechanical engineering, and control theory.

> **The Core Strategy**
>
> $$\underbrace{ax''+bx'+cx = f(t)}_{\text{ODE (hard to solve)}} \xrightarrow{\;\mathcal{L}\;}
> \underbrace{(as^2+bs+c)X(s) = F(s) + \text{ICs}}_{\text{algebraic equation (easy to solve)}}
> \xrightarrow{\;\mathcal{L}^{-1}\;} \underbrace{x(t)}_{\text{solution}}$$
>
> The Laplace transform converts differentiation into multiplication by
> $s$. What was a differential equation becomes an algebraic equation
> for $X(s)$. We solve algebraically, then invert.

Think of it as translating a hard problem in one language (differential
equations) into an easier problem in another language (algebra), solving
there, then translating back. The same philosophy underlies logarithms
(which convert multiplication into addition), the Fourier transform
(which converts convolution into multiplication), and many other
transforms throughout mathematics.

------------------------------------------------------------------------

## Definition

> **Definition: The Laplace Transform**
>
> Let $x = x(t)$ be defined on $[0, \infty)$. The **Laplace transform**
> of $x(t)$ is the function $X(s)$ defined by
> $$\mathcal{L}[x(t)](s) = X(s) = \int_0^\infty x(t)\,e^{-st}\,dt,$$
> provided the improper integral converges. We also write
> $\mathcal{L}[x] = X(s)$.

**Unpacking the definition.** The integrand $x(t)e^{-st}$ is a function
of both $t$ and $s$. We integrate over $t$ from $0$ to $\infty$, which
eliminates the $t$ dependence and leaves a function of $s$ alone. The
factor $e^{-st}$ is a **damping kernel** that forces the integrand to
zero as $t\to\infty$ (provided $s$ is large enough), making the improper
integral converge.

The variable $s$ lives in the **transform domain** (sometimes called the
frequency domain or the $s$-domain), while $t$ lives in the **time
domain**. We use uppercase letters for transform-domain functions:
$x(t) \leftrightarrow X(s)$.

> **Time Domain vs. Transform Domain**
>
> |          |    Time domain     |        Transform domain         |
> |----------|:------------------:|:-------------------------------:|
> | Variable |        $t$         |               $s$               |
> | Function | $x(t)$ (lowercase) |       $X(s)$ (uppercase)        |
> | Language |   “what happens”   | “how fast it oscillates/decays” |

------------------------------------------------------------------------

## Computing Transforms from the Definition

### Example 1 — $\mathcal{L}[e^{at}]$

$$X(s) = \int_0^\infty e^{at}e^{-st}\,dt = \int_0^\infty e^{(a-s)t}\,dt = \frac{1}{a-s}e^{(a-s)t}\Bigg|_{t=0}^{t=\infty}.$$

For the upper limit to vanish we need $a - s < 0$, i.e., $s > a$.
Evaluating explicitly:
$$\frac{1}{a-s}e^{(a-s)t}\Bigg|_{t=0}^{t=\infty} = \frac{1}{a-s}\bigl[0 - 1\bigr] = \frac{-1}{a-s} = \frac{1}{s-a}.$$

$$\boxed{\mathcal{L}[e^{at}] = \frac{1}{s-a}, \quad s > a.}$$

**Special case** $a=0$: $\mathcal{L}[1] = \frac{1}{s}$, $s > 0$.

### Example 2 — $\mathcal{L}[t]$

Integration by parts with $u = t$, $dv = e^{-st}dt$:

$$X(s) = \int_0^\infty t\,e^{-st}\,dt = \left[t\cdot\frac{e^{-st}}{-s}\right]_0^\infty + \frac{1}{s}\int_0^\infty e^{-st}\,dt = 0 + \frac{1}{s}\cdot\frac{1}{s} = \frac{1}{s^2}.$$

The boundary term $\bigl[te^{-st}/(-s)\bigr]_0^\infty = 0$: at $t=0$ the
term vanishes; as $t\to\infty$, $te^{-st}\to 0$ for $s>0$ (by
L’Hôpital’s rule,
$\lim_{t\to\infty} t/e^{st} = \lim_{t\to\infty} 1/(se^{st}) = 0$).

$$\boxed{\mathcal{L}[t] = \frac{1}{s^2}, \quad s > 0.}$$

The pattern continues: $\mathcal{L}[t^n] = n!/s^{n+1}$.

### Example 3 — $\mathcal{L}[\sin kt]$ and $\mathcal{L}[\cos kt]$

Using $\sin kt = \frac{1}{2i}(e^{ikt}-e^{-ikt})$ and the formula for
$\mathcal{L}[e^{at}]$:

$$\mathcal{L}[\sin kt] = \frac{1}{2i}\left(\frac{1}{s-ik}-\frac{1}{s+ik}\right) = \frac{1}{2i}\cdot\frac{2ik}{s^2+k^2} = \frac{k}{s^2+k^2}.$$

Similarly $\mathcal{L}[\cos kt] = s/(s^2+k^2)$.

> **A Pattern Worth Noting**
>
> Compare $\mathcal{L}[\sin kt] = k/(s^2+k^2)$ and
> $\mathcal{L}[\cos kt] = s/(s^2+k^2)$. The numerator of the cosine
> transform involves $s$ (the transform variable), while the sine
> transform has a constant numerator $k$. This asymmetry reflects the
> fact that $\cos(0)=1$ and $\sin(0)=0$ — the cosine has a nonzero
> initial value.

In [2]:
t_plot = np.linspace(0, 5, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: damped integrand for two values of s
x_func = np.sin(2*t_plot)
for s_val, color, lbl in [(0.5, 'steelblue', '$s=0.5$'), (2.0, 'crimson', '$s=2.0$')]:
    damp = np.exp(-s_val*t_plot)
    integrand = x_func * damp
    axes[0].plot(t_plot, integrand, color=color, lw=2, label=f'$\\sin(2t)e^{{-{s_val}t}}$ ({lbl})')
    axes[0].fill_between(t_plot, 0, integrand, where=(integrand>0), alpha=0.12, color=color)
    axes[0].fill_between(t_plot, 0, integrand, where=(integrand<0), alpha=0.12, color=color)

axes[0].plot(t_plot, x_func, 'k--', lw=1.2, alpha=0.5, label='$\\sin(2t)$ (undamped)')
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_xlabel('$t$'); axes[0].set_ylabel('integrand')
axes[0].set_title(r'Integrand $\sin(2t)\,e^{-st}$ for two values of $s$')
axes[0].legend(fontsize=8.5)

# Right: X(s) = 2/(s^2+4)
s_arr = np.linspace(0.01, 6, 400)
X_s   = 2/(s_arr**2 + 4)
axes[1].plot(s_arr, X_s, color='seagreen', lw=2.5, label=r'$X(s)=\frac{2}{s^2+4}$')
for s_val, color in [(0.5, 'steelblue'), (2.0, 'crimson')]:
    val = 2/(s_val**2+4)
    axes[1].plot(s_val, val, 'o', color=color, markersize=9,
                 label=f'$X({s_val})={val:.3f}$ (area under left plot)')
axes[1].set_xlabel('$s$'); axes[1].set_ylabel('$X(s)$')
axes[1].set_title(r'$\mathcal{L}[\sin 2t](s) = 2/(s^2+4)$')
axes[1].legend(fontsize=8.5); axes[1].set_ylim(0, 0.55)

plt.suptitle('How the Laplace transform works', fontsize=12)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

## Linearity

> **Theorem: Linearity**
>
> For any constants $\alpha$, $\beta$ and functions $x(t)$, $y(t)$:
> $$\mathcal{L}[\alpha x + \beta y] = \alpha\mathcal{L}[x] + \beta\mathcal{L}[y].$$

This follows immediately from the linearity of the integral and means we
can build up transforms of complicated functions from the table, one
term at a time.

**Example.**
$$\mathcal{L}[3e^{-2t} + 5\sin 4t] = 3\cdot\frac{1}{s+2} + 5\cdot\frac{4}{s^2+16} = \frac{3}{s+2} + \frac{20}{s^2+16}.$$

------------------------------------------------------------------------

## Existence of the Transform

Not every function has a Laplace transform. Two sufficient conditions
guarantee existence:

1.  **Piecewise continuity (PWC)**: on any bounded subinterval $[0,b]$,
    $x(t)$ has only finitely many jump discontinuities, and at each jump
    the one-sided limits exist (are finite).

2.  **Exponential order**: there exist constants $M > 0$ and $r$ such
    that $|x(t)| \leq Me^{rt}$ for all $t > t_0$.

If both conditions hold, then $X(s)$ exists for all $s > r$.

**Functions that fail:** - $x(t) = e^{t^2}$: grows faster than any
exponential, so not of exponential order. - $x(t) = 1/t$: not PWC on
$[0,b]$ (blows up at $t=0$). - $X(s) = (1/s)e^s$: grows without bound as
$s\to\infty$ (whereas every valid Laplace transform of a
piecewise-continuous, exponentially-ordered function satisfies
$X(s)\to 0$ as $s\to\infty$), so $(1/s)e^s$ cannot correspond to any
such function.

------------------------------------------------------------------------

## The Inverse Transform

Given $X(s)$, we want the function $x(t)$ whose transform is $X(s)$. We
write: $$x(t) = \mathcal{L}^{-1}[X(s)].$$

The inverse transform is also linear:
$\mathcal{L}^{-1}[\alpha X + \beta Y] = \alpha\mathcal{L}^{-1}[X] + \beta\mathcal{L}^{-1}[Y]$.

**In practice**, we find inverse transforms by: 1. Looking up the form
in the table, OR 2. Using **partial fractions** to decompose $X(s)$ into
table-recognizable pieces.

### Partial Fractions — The Key Algebraic Skill

Most $X(s)$ arising from ODEs are rational functions. The partial
fractions decomposition rewrites them as sums of simpler fractions, each
of which appears in the table.

**Type 1 — Distinct real poles.** If $X(s) = P(s)/[(s-a)(s-b)]$:
$$X(s) = \frac{A}{s-a} + \frac{B}{s-b} \;\Longrightarrow\; x(t) = Ae^{at} + Be^{bt}.$$

**Type 2 — Repeated real pole.** If $X(s) = P(s)/(s-a)^2$:
$$X(s) = \frac{A}{s-a} + \frac{B}{(s-a)^2} \;\Longrightarrow\; x(t) = Ae^{at} + Bte^{at}.$$

**Type 3 — Complex conjugate poles.** If
$X(s) = P(s)/[(s-\alpha)^2+\beta^2]$:

Complete the square in the denominator, then match to
$\mathcal{L}[e^{\alpha t}\sin\beta t]$ and
$\mathcal{L}[e^{\alpha t}\cos\beta t]$.

**Worked example.** Find
$\mathcal{L}^{-1}\!\left[\dfrac{2s+3}{s^2+2s+5}\right]$.

Complete the square: $s^2+2s+5 = (s+1)^2+4$. Write the numerator to
match the shifted cosine and sine forms:
$$\frac{2s+3}{(s+1)^2+4} = \frac{2(s+1)-2+3}{(s+1)^2+4} = \frac{2(s+1)}{(s+1)^2+4} + \frac{1}{(s+1)^2+4}.$$

The second fraction needs a factor of $\beta = 2$ in the numerator:
$\dfrac{1}{(s+1)^2+4} = \dfrac{1}{2}\cdot\dfrac{2}{(s+1)^2+4}$.

Reading off from the table with $\alpha=-1$, $\beta=2$:
$$x(t) = 2e^{-t}\cos(2t) + \tfrac{1}{2}e^{-t}\sin(2t).$$

**Example.** Find
$\mathcal{L}^{-1}\!\left[\frac{2s+9}{(s+1)(s+3)}\right]$.

$$\frac{2s+9}{(s+1)(s+3)} = \frac{A}{s+1}+\frac{B}{s+3}.$$

Cover-up rule: $A = (2(-1)+9)/(-1+3) = 7/2$;
$B = (2(-3)+9)/(-3+1) = -3/2$.

$$x(t) = \frac{7}{2}e^{-t} - \frac{3}{2}e^{-3t}.$$

In [3]:
s_v = sym.Symbol('s')
t_v = sym.Symbol('t', positive=True)

# Demonstrate partial fractions with SymPy
examples_inv = [
    (r"\frac{2s+9}{(s+1)(s+3)}",          (2*s_v+9)/((s_v+1)*(s_v+3))),
    (r"\frac{3}{s^2-4}",                    3/(s_v**2-4)),
    (r"\frac{2s+3}{s^2+2s+5}",             (2*s_v+3)/(s_v**2+2*s_v+5)),
    (r"\frac{1}{s(s+2)}",                   1/(s_v*(s_v+2))),
]

for label, expr in examples_inv:
    pf = sym.apart(expr, s_v)
    inv = sym.inverse_laplace_transform(expr, s_v, t_v)
    print(f"$X(s) = {label}$")
    display(Math(r"\text{PF: } " + sym.latex(pf)))
    display(Math(r"x(t) = " + sym.latex(sym.simplify(inv))))
    print()

$X(s) = \frac{2s+9}{(s+1)(s+3)}$


$X(s) = \frac{3}{s^2-4}$


$X(s) = \frac{2s+3}{s^2+2s+5}$


$X(s) = \frac{1}{s(s+2)}$

------------------------------------------------------------------------

## The Transform of a Derivative — The ODE Connection

The single most important property for solving ODEs is:

> **Transform of Derivatives**
>
> $$\mathcal{L}[x'(t)] = s\,X(s) - x(0)$$
> $$\mathcal{L}[x''(t)] = s^2 X(s) - s\,x(0) - x'(0)$$
>
> More generally:
> $\mathcal{L}[x^{(n)}] = s^n X(s) - s^{n-1}x(0) - \cdots - x^{(n-1)}(0)$.

**Derivation** (for $x'$): integrate by parts with $u = e^{-st}$,
$dv = x'(t)dt$:
$$\mathcal{L}[x'] = \int_0^\infty x'(t)e^{-st}dt = \left[x(t)e^{-st}\right]_0^\infty + s\int_0^\infty x(t)e^{-st}dt = -x(0) + sX(s).$$

> **Why This Transforms an ODE Into Algebra**
>
> Applying $\mathcal{L}$ to $ax''+bx'+cx = f(t)$ and using linearity:
> $$a\mathcal{L}[x''] + b\mathcal{L}[x'] + c\mathcal{L}[x] = \mathcal{L}[f].$$
> Substituting the derivative formulas:
> $$\bigl(as^2+bs+c\bigr)X(s) = F(s) + a\bigl(sx(0)+x'(0)\bigr) + b\,x(0).$$
> The initial conditions are **automatically incorporated** — no
> separate step needed. Solving for $X(s)$ is pure algebra.

> **Note**
>
> **Connection to Notes 7.** These derivative formulas are restated and
> proved formally as **Theorem 3.11** in Notes 7 (equations (3.6) and
> (3.7) of Logan). Notes 7 also shows how the same formulas can be used
> in reverse — as a *bootstrapping* technique for finding transforms of
> new functions such as $\mathcal{L}[t^n]$ and $\mathcal{L}[\cosh t]$
> without evaluating the defining integral.

### Full ODE Example

**Solve** $x'' + 4x' + 3x = 0$, $x(0) = 2$, $x'(0) = 1$.

**Step 1.** Apply $\mathcal{L}$:
$$[s^2X - 2s - 1] + 4[sX - 2] + 3X = 0$$ Collecting the $X$ terms on the
left and moving IC terms to the right:
$$(s^2+4s+3)X = 2s + 1 + 8 = 2s+9.$$

**Step 2.** Solve for $X(s)$: $$X(s) = \frac{2s+9}{(s+1)(s+3)}.$$

**Step 3.** Partial fractions:
$$X(s) = \frac{7/2}{s+1} + \frac{-3/2}{s+3}.$$

**Step 4.** Invert term by term:
$$\boxed{x(t) = \frac{7}{2}e^{-t} - \frac{3}{2}e^{-3t}.}$$

In [4]:
t_plot = np.linspace(0, 4, 300)
x_exact = 3.5*np.exp(-t_plot) - 1.5*np.exp(-3*t_plot)

def ode_ex1(t, y): return [y[1], -4*y[1] - 3*y[0]]
sol = solve_ivp(ode_ex1, (0, 4), [2.0, 1.0], dense_output=True, max_step=0.02)
t_dots = np.linspace(0, 4, 20)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_plot, x_exact, color='steelblue', lw=2.5, label=r'$\frac{7}{2}e^{-t}-\frac{3}{2}e^{-3t}$')
ax.plot(t_dots, sol.sol(t_dots)[0], 'ro', markersize=6, label='Numerical (solve_ivp)')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('$t$'); ax.set_ylabel('$x(t)$')
ax.set_title(r"$x''+4x'+3x=0$, $x(0)=2$, $x'(0)=1$")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

## The Heaviside Function — Modeling Switches

Many real forcing functions are **piecewise defined**: a voltage
switched on at $t=3$, a force applied for exactly $2$ seconds, a pump
turned off at $t=5$. The **[Heaviside
function](https://en.wikipedia.org/wiki/Heaviside_step_function)** is
the mathematical on/off switch:

$$H(t-a) = \begin{cases} 0, & t < a \\ 1, & t \geq a. \end{cases}$$

Its Laplace transform is:
$$\mathcal{L}[H(t-a)] = \int_a^\infty e^{-st}\,dt = \frac{e^{-as}}{s}, \quad s > 0.$$

### Building Piecewise Functions from Heaviside Steps

Any piecewise-constant function can be written as a sum of Heaviside
steps. The key insight is that $H(t-a) - H(t-b)$ equals $1$ only on the
interval $[a,b)$ and $0$ elsewhere — it is a **pulse**.

**Example** (Logan §3.1, Example 3.8). The function
$$f(t) = \begin{cases} 3, & 0 \leq t < 2 \\ 4, & 2 \leq t < 3 \\ 2, & 3 \leq t < 6 \\ 0, & t > 6 \end{cases}$$
can be written as:
$$f(t) = 3H(t) + 1\cdot H(t-2) - 2\cdot H(t-3) - 2\cdot H(t-6).$$

Each term “adjusts” the running value at the moment of the jump:

| Term             | What it does                     | At $t=$ |
|------------------|----------------------------------|---------|
| $3H(t)$          | Switches on value $3$            | $0$     |
| $+1\cdot H(t-2)$ | Adds $1$ (making value $4$)      | $2$     |
| $-2\cdot H(t-3)$ | Subtracts $2$ (making value $2$) | $3$     |
| $-2\cdot H(t-6)$ | Subtracts $2$ (making value $0$) | $6$     |

> **Tip**
>
> Think of this as a **running adjustment**: start at $3$, add $1$ at
> $t=2$, subtract $2$ at $t=3$, subtract $2$ at $t=6$. Each Heaviside
> term corrects the running value by exactly the size of the jump at
> that moment. The coefficients are always the *differences* between
> adjacent constant values, not the values themselves.

By linearity:
$$F(s) = \frac{3}{s} + \frac{e^{-2s}}{s} - \frac{2e^{-3s}}{s} - \frac{2e^{-6s}}{s} = \frac{1}{s}\!\left(3 + e^{-2s} - 2e^{-3s} - 2e^{-6s}\right).$$

In [5]:
t_plot = np.linspace(-0.3, 8, 600)
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# Piecewise function
def f_piece(tv):
    return np.where(tv<0, 0,
           np.where(tv<2, 3,
           np.where(tv<3, 4,
           np.where(tv<6, 2, 0))))

axes[0].step(t_plot, f_piece(t_plot), where='post', color='k', lw=2.5,
             label='$f(t)$', zorder=5)

# Individual Heaviside contributions
H = lambda tv, a: np.where(tv >= a, 1.0, 0.0)
contribs = [
    (3*H(t_plot,0),      'steelblue',  '+3$H(t)$'),
    (1*H(t_plot,2),      'darkorange', '+1$H(t-2)$'),
    (-2*H(t_plot,3),     'seagreen',   '-2$H(t-3)$'),
    (-2*H(t_plot,6),     'crimson',    '-2$H(t-6)$'),
]
running = np.zeros_like(t_plot)
for contrib, color, lbl in contribs:
    axes[0].fill_between(t_plot, running, running+contrib, alpha=0.25, color=color)
    running += contrib

for _, color, lbl in contribs:
    axes[0].plot([], [], color=color, linewidth=8, alpha=0.4, label=lbl)

axes[0].set_xlabel('$t$'); axes[0].set_ylabel('$f(t)$')
axes[0].set_title('Piecewise function as Heaviside steps')
axes[0].legend(fontsize=9, ncol=4)
axes[0].set_xlim(-0.3, 8); axes[0].set_ylim(-0.5, 5)

# Laplace transform F(s)
s_arr = np.linspace(0.01, 4, 400)
F_s = (3 + np.exp(-2*s_arr) - 2*np.exp(-3*s_arr) - 2*np.exp(-6*s_arr))/s_arr
axes[1].plot(s_arr, F_s, color='seagreen', lw=2.5)
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_xlabel('$s$'); axes[1].set_ylabel('$F(s)$')
axes[1].set_title(r'Laplace transform $F(s) = \frac{1}{s}(3+e^{-2s}-2e^{-3s}-2e^{-6s})$')

plt.tight_layout()
plt.show()

------------------------------------------------------------------------

## Two Key Operational Properties

### The Shift Property

> **Shift Property**
>
> $$\mathcal{L}[f(t)e^{at}] = F(s-a),$$ where
> $F(s) = \mathcal{L}[f(t)]$.
>
> Multiplying a time-domain function by $e^{at}$ **shifts its
> transform** $a$ units to the right in the $s$-domain.

This is proved directly from the definition:
$\int_0^\infty f(t)e^{at}e^{-st}dt = \int_0^\infty f(t)e^{-(s-a)t}dt = F(s-a)$.

**Examples.**

| $f(t)$    | $F(s)$      | $f(t)e^{at}$    | $\mathcal{L}[f(t)e^{at}]$ |
|-----------|-------------|-----------------|---------------------------|
| $t$       | $1/s^2$     | $te^{-2t}$      | $1/(s+2)^2$               |
| $\sin 3t$ | $3/(s^2+9)$ | $e^{-t}\sin 3t$ | $3/((s+1)^2+9)$           |
| $\cos 3t$ | $s/(s^2+9)$ | $e^{2t}\cos 3t$ | $(s-2)/((s-2)^2+9)$       |
| $t^2$     | $2/s^3$     | $t^2 e^{5t}$    | $2/(s-5)^3$               |

The shift property is essential for inverting transforms with
**completed-square denominators** like $(s+1)^2+4$.

### The Switching Property

> **Switching Property**
>
> $$\mathcal{L}[H(t-a)f(t-a)] = e^{-as}F(s).$$ Equivalently (inverse
> form): $$\mathcal{L}^{-1}[e^{-as}F(s)] = H(t-a)f(t-a).$$
>
> A factor of $e^{-as}$ in the transform domain corresponds to a **time
> delay of $a$** in the time domain.

**Intuition:** $H(t-a)f(t-a)$ is the function $f(t)$ “switched on” at
$t=a$ (it is zero before $a$ and equals $f(t-a)$ after $a$). The factor
$e^{-as}$ encodes this delay.

> **Tip**
>
> **How to apply the switching property in reverse.** When inverting
> $X(s)$, if you see an $e^{-as}$ factor multiplying some $F(s)$: 1.
> **Strip** the $e^{-as}$ factor and identify $F(s)$. 2. **Invert**
> $F(s)$ alone to get $f(t) = \mathcal{L}^{-1}[F(s)]$. 3. **Replace**
> $t$ with $t-a$ in $f(t)$. 4. **Multiply** by $H(t-a)$.
>
> The result is $H(t-a)f(t-a)$.

**Example.** Find $\mathcal{L}^{-1}\!\left[\frac{e^{-3s}}{s-2}\right]$.

We know $\mathcal{L}^{-1}[1/(s-2)] = e^{2t}$, so $F(s) = 1/(s-2)$,
$f(t) = e^{2t}$. By the switching property:
$$\mathcal{L}^{-1}\!\left[\frac{e^{-3s}}{s-2}\right] = H(t-3)\,e^{2(t-3)}.$$
The exponential grows from $t=3$ onward (not from $t=0$).

In [6]:
s_v = sym.Symbol('s')
t_v = sym.Symbol('t', positive=True)

print("=== Shift property examples (SymPy verification) ===")
for f_t, lbl in [(t_v*sym.exp(-2*t_v), 'te^{-2t}'),
                  (sym.exp(-t_v)*sym.sin(3*t_v), 'e^{-t}sin(3t)'),
                  (sym.exp(2*t_v)*sym.cos(3*t_v), 'e^{2t}cos(3t)')]:
    F = sym.laplace_transform(f_t, t_v, s_v, noconds=True)
    display(Math(r"\mathcal{L}[" + lbl + r"] = " + sym.latex(sym.simplify(F))))

=== Shift property examples (SymPy verification) ===

------------------------------------------------------------------------

## Application: Discontinuous Forcing

The true power of the Laplace transform appears when $f(t)$ is
discontinuous. Methods from Chapter 2 are helpless here; the Laplace
transform handles it in stride.

**Example.** Solve the RC circuit equation
$$x' + 2x = f(t), \quad x(0) = 0,$$ where $f(t) = 1$ for $0 \leq t < 3$
and $f(t) = 0$ for $t \geq 3$ (voltage switched off at $t = 3$).

**Step 1.** Write $f(t) = H(t) - H(t-3)$, so $F(s) = 1/s - e^{-3s}/s$.

**Step 2.** Transform the ODE:
$$(s+2)X(s) = \frac{1}{s} - \frac{e^{-3s}}{s} \;\Longrightarrow\; X(s) = \frac{1}{s(s+2)} - \frac{e^{-3s}}{s(s+2)}.$$

**Step 3.** Partial fractions:
$\dfrac{1}{s(s+2)} = \dfrac{1/2}{s} - \dfrac{1/2}{s+2}$, so

$$\mathcal{L}^{-1}\!\left[\frac{1}{s(s+2)}\right] = \frac{1}{2}(1 - e^{-2t}).$$

**Step 4.** Apply the switching property to the second term:

$$\mathcal{L}^{-1}\!\left[\frac{e^{-3s}}{s(s+2)}\right] = H(t-3)\cdot\frac{1}{2}(1-e^{-2(t-3)}).$$

**Solution:**
$$\boxed{x(t) = \frac{1}{2}(1-e^{-2t}) - \frac{1}{2}H(t-3)(1-e^{-2(t-3)}).}$$

In [7]:
t_plot = np.linspace(0, 8, 600)
H = lambda tv, a: np.where(tv >= a, 1.0, 0.0)

x_exact = 0.5*(1 - np.exp(-2*t_plot)) - 0.5*H(t_plot, 3)*(1 - np.exp(-2*(t_plot-3)))
f_plot  = H(t_plot, 0) - H(t_plot, 3)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

axes[0].step(t_plot, f_plot, where='post', color='crimson', lw=2.5)
axes[0].set_ylabel('$f(t)$'); axes[0].set_ylim(-0.2, 1.4)
axes[0].set_title('Forcing $f(t) = H(t)-H(t-3)$')
axes[0].fill_between(t_plot, 0, f_plot, alpha=0.2, color='crimson')

axes[1].plot(t_plot, x_exact, color='steelblue', lw=2.5, label='Analytical (Laplace)')
def rhs(tv, y): return [(-H(tv,0)-H(tv,3))*0 + (1 if tv<3 else 0) - 2*y[0]]
sol_num = solve_ivp(lambda tv, y: [(1 if tv<3 else 0) - 2*y[0]],
                   (0, 8), [0.0], dense_output=True, max_step=0.02)
t_dots = np.linspace(0, 8, 25)
axes[1].plot(t_dots, sol_num.sol(t_dots)[0], 'ro', markersize=6, label='Numerical')
axes[1].axhline(0.5, color='gray', ls=':', lw=1.2, label='Steady state $x=0.5$')
axes[1].axvline(3, color='gray', ls='--', lw=1.2, alpha=0.7)
axes[1].set_xlabel('$t$'); axes[1].set_ylabel('$x(t)$')
axes[1].set_title("Solution via Laplace transform")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

> **Note**
>
> **This example is a first taste of discontinuous forcing** — the
> forcing is piecewise-constant and the ODE is first-order. **Notes 7**
> takes up substantially harder cases: non-constant piecewise pieces
> (e.g., $f(t)=t$ on one interval then $f(t)=2$ on another),
> second-order oscillator equations, and RC circuits driven by square
> pulses. The Heaviside and switching tools developed here carry over
> directly; only the algebra becomes more involved.

------------------------------------------------------------------------

## The Short Table of Laplace Transforms

The following table (from Logan (2015), Table 3.1) collects the
transforms needed throughout this chapter. The left column lists
time-domain functions $x(t)$ and the right column lists their transforms
$X(s) = \mathcal{L}[x]$.

| $x(t)$ | $X(s) = \mathcal{L}[x]$ | Notes |
|------------------------|------------------------|------------------------|
| $e^{at}$ | $\dfrac{1}{s-a}$, $\;s>a$ |  |
| $t^n$, $n=0,1,2,\ldots$ | $\dfrac{n!}{s^{n+1}}$, $\;s>0$ | $n=0$: $\mathcal{L}[1]=1/s$ |
| $f(t)e^{at}$ | $F(s-a)$ | Shift property |
| $H(t-a)$ | $\dfrac{1}{s}e^{-as}$, $\;s>0$ |  |
| $H(t-a)f(t-a)$ | $e^{-as}F(s)$ | Switching property |
| $f(t)H(t-a)$ | $e^{-as}\mathcal{L}[f(t+a)]$ | Use when $f$ is *not* shifted (see note below) |
| $\sin kt$ | $\dfrac{k}{s^2+k^2}$ |  |
| $\cos kt$ | $\dfrac{s}{s^2+k^2}$ |  |
| $\sinh kt$ | $\dfrac{k}{s^2-k^2}$ |  |
| $\cosh kt$ | $\dfrac{s}{s^2-k^2}$ |  |
| $e^{at}\sin kt$ | $\dfrac{k}{(s-a)^2+k^2}$ | Shift applied to $\sin$ |
| $e^{at}\cos kt$ | $\dfrac{s-a}{(s-a)^2+k^2}$ | Shift applied to $\cos$ |
| $t\sin kt$ | $\dfrac{2ks}{(s^2+k^2)^2}$ |  |
| $t\cos kt$ | $\dfrac{s^2-k^2}{(s^2+k^2)^2}$ |  |
| $\dfrac{1}{a-b}(e^{at}-e^{bt})$ | $\dfrac{1}{(s-a)(s-b)}$ | $a\neq b$; covers PF |
| $x^{(n)}(t)$ | $s^n X(s) - s^{n-1}x(0) - \cdots - x^{(n-1)}(0)$ | Key ODE formula |
| $f(at)$ | $\dfrac{1}{a}F\!\left(\dfrac{s}{a}\right)$, $\;a>0$ | Scaling |
| $t^a$ | $\dfrac{\Gamma(a+1)}{s^{a+1}}$, $\;s>0$ | Non-integer powers |
| $\delta_a(t)$ | $e^{-as}$ | Dirac delta at $t=a$ |
| $\delta_0(t)$ | $1$ | Unit impulse |
| $\displaystyle\int_0^t f(\tau)g(t-\tau)\,d\tau$ | $F(s)G(s)$ | Convolution theorem |
| $\displaystyle\int_0^t f(\tau)\,d\tau$ | $\dfrac{1}{s}F(s)$ |  |
| $t\,f(t)$ | $-F'(s)$ |  |
| $t^n f(t)$ | $(-1)^n F^{(n)}(s)$ |  |

> **Note**
>
> **Two Heaviside entries: which to use?** The switching property
> $H(t-a)f(t-a)\leftrightarrow e^{-as}F(s)$ is the standard form — use
> it whenever the function is already written in the shifted form
> $f(t-a)$. The second entry
> $f(t)H(t-a)\leftrightarrow e^{-as}\mathcal{L}[f(t+a)]$ arises when
> $f(t)$ is *not* shifted (e.g., $t^2 H(t-3)$ rather than
> $(t-3)^2 H(t-3)$) — you must first evaluate $\mathcal{L}[f(t+a)]$ and
> attach $e^{-as}$. In most problems, rewriting $f(t)$ in shifted form
> and using the standard entry is cleaner.

In [8]:
# Verify a selection of table entries with SymPy
s_v = sym.Symbol('s', positive=True)
t_v = sym.Symbol('t', positive=True)
k_v = sym.Symbol('k', positive=True)
a_v = sym.Symbol('a', positive=True)

entries = [
    (sym.sin(k_v*t_v), f'sin(kt)',     k_v/(s_v**2+k_v**2)),
    (sym.cos(k_v*t_v), f'cos(kt)',     s_v/(s_v**2+k_v**2)),
    (sym.exp(a_v*t_v)*sym.sin(k_v*t_v), 'e^{at}sin(kt)',
     k_v/((s_v-a_v)**2+k_v**2)),
    (t_v*sym.sin(k_v*t_v), 't*sin(kt)',
     2*k_v*s_v/(s_v**2+k_v**2)**2),
]

print("Spot-checking table entries via SymPy:")
for f_t, name, F_expected in entries:
    F_computed = sym.laplace_transform(f_t, t_v, s_v, noconds=True)
    F_diff = sym.simplify(F_computed - F_expected)
    status = "✓" if F_diff == 0 else f"Diff={F_diff}"
    print(f"  {name}: {status}")

Spot-checking table entries via SymPy:
  sin(kt): ✓
  cos(kt): ✓
  e^{at}sin(kt): ✓
  t*sin(kt): ✓

------------------------------------------------------------------------

## Summary of the Method

The complete workflow for solving $ax''+bx'+cx = f(t)$, $x(0)=x_0$,
$x'(0)=x_1$ via Laplace transforms:

> **Note**
>
> **Relationship to Notes 7.** Notes 7 presents the same procedure as
> the **Three-Step Method** (following Logan’s Figure 3.2): transform →
> solve algebraically → invert. The five-item checklist below is the
> same algorithm with each stage broken into more explicit sub-steps.
> The two descriptions are complementary: the three-step framing names
> the logical stages; this checklist gives the implementation details
> within each stage.

> **The Five Steps**
>
> 1.  **Transform both sides**: apply $\mathcal{L}$ to the ODE. Use the
>     derivative formula to handle $\mathcal{L}[x'']$ and
>     $\mathcal{L}[x']$. Rewrite $f(t)$ using Heaviside steps if
>     piecewise.
>
> 2.  **Incorporate ICs**: the derivative formula automatically brings
>     in $x(0)$ and $x'(0)$.
>
> 3.  **Solve for $X(s)$**: this is pure algebra — no differential
>     equations.
>
> 4.  **Partial fractions**: decompose $X(s)$ into standard forms from
>     the table.
>
> 5.  **Invert**: apply $\mathcal{L}^{-1}$ term by term; use the
>     switching property for any $e^{-as}$ factors.

| Step | What you do | Mathematics involved |
|----|----|----|
| 1 | $\mathcal{L}[\text{ODE}]$ | Integral transforms, linearity |
| 2 | Plug in ICs | The formula $\mathcal{L}[x''] = s^2X - sx_0 - x_1$ |
| 3 | Solve for $X(s)$ | Algebra |
| 4 | Partial fractions | Algebra (factoring, cover-up rule) |
| 5 | $\mathcal{L}^{-1}$, switching | Table lookup |

> **Common Traps**
>
> - **Time delays:** every $e^{-as}$ factor in $X(s)$ signals a time
>   delay. Strip it, invert the remaining $F(s)$ to get $f(t)$, then
>   replace $t\to t-a$ and multiply by $H(t-a)$.
> - **Complex poles:** when the denominator has the form
>   $(s-\alpha)^2+\beta^2$, complete the square first. Then split the
>   numerator to match the $e^{\alpha t}\cos\beta t$ and
>   $e^{\alpha t}\sin\beta t$ table entries (see the Type 3 example in
>   the Partial Fractions section above).
> - **Repeated poles:** a factor $(s-a)^2$ in the denominator
>   corresponds to $Ae^{at}+Bte^{at}$ in the time domain, not just
>   $Ae^{at}$. Use both the $1/(s-a)$ and $1/(s-a)^2$ terms in the
>   partial fractions decomposition.

These notes are also viewable as a slide deck presentation:
<a href="https://math341diffeq.netlify.app/notes_06_slides.html"
target="_blank">Open slides in full screen</a>

> **Looking Ahead**
>
> The transforms and properties developed here are the foundation, but
> the real payoff comes in §3.2 and beyond. The **convolution theorem**
> ($\mathcal{L}[f*g] = F(s)G(s)$, already in the table) will allow us to
> express solutions as integrals — a powerful tool for systems with
> arbitrary forcing. The **Dirac delta** $\delta_a(t)$ (also in the
> table, with $\mathcal{L}[\delta_a] = e^{-as}$) models instantaneous
> impulses — hammer blows, lightning strikes, and analogous events in
> circuits — that no classical function can describe. These two tools
> complete the Laplace transform toolkit.

> **Note**
>
> **Next:** Applying Laplace Transforms to Differential Equations —
> Logan §3.2.

------------------------------------------------------------------------

## Relevant Videos

### What is a Laplace Transform?:

### Intro to Laplace Transforms:

### Properties of Laplace Transforms:

------------------------------------------------------------------------

## References

Logan, J David. 2015. *A First Course in Differential Equations, Third
Edition*.

> **Session Info**
>
> ``` python
> import sys
> print("Python:", sys.version)
> print('\n'.join(f'{m.__name__}=={m.__version__}' for m in globals().values() if getattr(m,'__version__',None)))
> ```
>
>     Python: 3.14.4 | packaged by conda-forge | (main, Apr  8 2026, 02:33:53) [Clang 20.1.8 ]
>     numpy==2.4.3
>     sympy==1.14.0
>     matplotlib==3.10.8

## Reuse

[![](http://mirrors.creativecommons.org/presskit/buttons/88x31/png/by-nc-sa.png?raw=1)](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode)

[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)